# Explore STEAD waveforms

Quick look at Noise vs Earthquake traces and the 10-second windows used for training.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
from src.stead_io import iter_subsample_traces
from src.utils import STEAD_SUBSAMPLE_DIR, SAMPLE_RATE_HZ
from src.windows import extract_earthquake_window, extract_noise_window

In [ ]:
eq = next(r for r in iter_subsample_traces(STEAD_SUBSAMPLE_DIR, split='test', max_earthquake=5, max_noise=0) if r.category != 'noise')
noise = next(r for r in iter_subsample_traces(STEAD_SUBSAMPLE_DIR, split='test', max_earthquake=0, max_noise=5) if r.category == 'noise')
print(eq.name, 'P=', eq.p_arrival, 'S=', eq.s_arrival)
print(noise.name)

In [ ]:
t = np.arange(eq.waveform.shape[0]) / SAMPLE_RATE_HZ
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, eq.waveform[:, 2], lw=0.6, color='black')
ax.axvline(eq.p_arrival / SAMPLE_RATE_HZ, color='tab:blue', label='P')
ax.axvline(eq.s_arrival / SAMPLE_RATE_HZ, color='tab:red', label='S')
ax.set_title(f'Earthquake Z channel — {eq.name}')
ax.set_xlabel('Time (s)'); ax.legend(); ax.grid(alpha=0.3)
plt.show()

In [ ]:
rng = np.random.default_rng(0)
ew = extract_earthquake_window(eq, rng=rng)
nw = extract_noise_window(noise, rng=rng)
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
axes[0].plot(ew.waveform[2], lw=0.8); axes[0].set_title('EQ window (Z, z-scored)')
axes[1].plot(nw.waveform[2], lw=0.8); axes[1].set_title('Noise window (Z, z-scored)')
plt.show()